# Omnimodal assistant with Qwen3-Omni and OpenVINO

Qwen3-Omni is the natively end-to-end multilingual omni-modal foundation models. It processes text, images, audio, and video, and delivers real-time streaming responses in both text and natural speech. We introduce several architectural upgrades to improve performance and efficiency. Key features:

* **State-of-the-art across modalities**: Early text-first pretraining and mixed multimodal training provide native multimodal support. While achieving strong audio and audio-video results, unimodal text and image performance does not regress. Reaches SOTA on 22 of 36 audio/video benchmarks and open-source SOTA on 32 of 36; ASR, audio understanding, and voice conversation performance is comparable to Gemini 2.5 Pro.

* **Multilingual**: Supports 119 text languages, 19 speech input languages, and 10 speech output languages.
  - **Speech Input**: English, Chinese, Korean, Japanese, German, Russian, Italian, French, Spanish, Portuguese, Malay, Dutch, Indonesian, Turkish, Vietnamese, Cantonese, Arabic, Urdu.
  - **Speech Output**: English, Chinese, French, German, Russian, Italian, Spanish, Portuguese, Japanese, Korean.

* **Novel Architecture**: MoE-based Thinker–Talker design with AuT pretraining for strong general representations, plus a multi-codebook design that drives latency to a minimum.

* **Real-time Audio/Video Interaction**: Low-latency streaming with natural turn-taking and immediate text or speech responses.

* **Flexible Control**: Customize behavior via system prompts for fine-grained control and easy adaptation.

* **Detailed Audio Captioner**: Qwen3-Omni-30B-A3B-Captioner is now open source: a general-purpose, highly detailed, low-hallucination audio captioning model that fills a critical gap in the open-source community.

<p align="center">
    <img src="https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-Omni/q3o_introduction.png" width="90%"/>
<p>

More details about model can be found in [model card](https://huggingface.co/Qwen/Qwen3-Omni-30B-A3B-Instruct) and original [repo](https://github.com/QwenLM/Qwen3-Omni/tree/main).

In this tutorial we consider how to convert and optimize Qwen3-Omni model for creating omnimodal chatbot. Additionally, we demonstrate how to apply stateful transformation on LLM part and model optimization techniques like weights compression using [NNCF](https://github.com/openvinotoolkit/nncf)

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert model to OpenVINO Intermediate Representation](#Convert-model-to-OpenVINO-Intermediate-Representation)
    - [Compress Language Model Weights to 4 bits](#Compress-Language-Model-Weights-to-4-bits)
- [Prepare model inference pipeline](#Prepare-model-inference-pipeline)
    - [Select device](#Select-device)
    - [Initialize model tasks](#Initialize-model-tasks)
- [Run OpenVINO model inference](#Run-OpenVINO-model-inference)
    - [Text-only input and Audio output](#Text-only-input-and-Audio-output)
    - [Text-Image input](#Text-Image-input)
    - [Audio-Text input](#Audio-Text-input)
    - [Video-text input](#Video-text-input)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [5]:
import requests
from pathlib import Path
import sys


if not Path("qwen_3_omni_moe_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/qwen3-omni-chatbot/qwen_3_omni_moe_helper.py")
    open("qwen_3_omni_moe_helper.py", "w").write(r.text)


if not Path("gradio_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/qwen3-omni-chatbot/gradio_helper.py")
    open("gradio_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

from pip_helper import pip_install


if sys.platform == "darwin":
    pip_install(
        "-q",
        "transformers==4.57.0",
        "torch==2.9",
        "torchvision==0.24.0",
        "accelerate",
        "qwen-omni-utils",
        "gradio>=4.19",
        "--no-cache-dir",
        "--extra-index-url",
        "https://download.pytorch.org/whl/cpu",
    )
else:
    pip_install(
        "-q",
        "transformers==4.57.0",
        "torch==2.9",
        "torchvision==0.24.0",
        "accelerate",
        "qwen-omni-utils[decord]",
        "gradio>=4.19",
        "--no-cache-dir",
        "--extra-index-url",
        "https://download.pytorch.org/whl/cpu",
    )
pip_install("openvino>=2025.4.0", "nncf>=2.19.0")

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("qwen3-omni-chatbot.ipynb")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-llms-openvino 0.5.1 requires llama-index-core<0.15,>=0.13.0, but you have llama-index-core 0.12.52.post1 which is incompatible.
llama-index-llms-openvino 0.5.1 requires llama-index-llms-huggingface<0.7,>=0.6.0, but you have llama-index-llms-huggingface 0.4.2 which is incompatible.
optimum-intel 1.27.0.dev0+f42cc42 requires transformers<4.56,>=4.45, but you have transformers 4.57.0 which is incompatible.
optimum-onnx 0.0.3 requires transformers<4.56.0,>=4.36.0, but you have transformers 4.57.0 which is incompatible.
torchaudio 2.7.1+cpu requires torch==2.7.1, but you have torch 2.9.0+cpu which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


## Convert model to OpenVINO Intermediate Representation
[back to top ⬆️](#Table-of-contents:)

OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). [OpenVINO model conversion API](https://docs.openvino.ai/2024/openvino-workflow/model-preparation.html#convert-a-model-with-python-convert-model) should be used for these purposes. `ov.convert_model` function accepts original PyTorch model instance and example input for tracing and returns `ov.Model` representing this model in OpenVINO framework. Converted model can be used for saving on disk using `ov.save_model` function or directly loading on device using `core.complie_model`.

`qwen_3_omni_moe_helper.py` script contains helper function for model conversion, please check its content if you interested in conversion details.

### Qwen3-Omni-MoE Architecture Overview

**Qwen3-Omni-MoE** is a multimodal Mixture-of-Experts (MoE) model capable of processing text, images, video, and audio inputs, and generating both text and speech outputs. The architecture consists of two main components: **Thinker** and **Talker**.

#### 1. Thinker (Understanding Module)

The Thinker is responsible for understanding multimodal inputs and generating semantic representations.

**Sub-models:**

- **Thinker Audio Encoder**: Processes audio features through convolutional layers (conv2d1, conv2d2, conv2d3) and extracts audio embeddings
- **Thinker Vision Encoder**: Processes images/videos using Vision Transformer with rotary positional embeddings (Qwen2-VL style)
- **Thinker Vision Positional Encoder**: Computes 3D rope indices for spatial-temporal features
- **Thinker Vision Merger**: Merges multi-scale visual features using spatial merge strategy
- **Thinker Embedding**: Text token embeddings for language input
- **Thinker Language Model**: MoE-based decoder with sparse experts (Qwen3MoeThinkerTextExperts) that processes fused multimodal embeddings and generates hidden states
- **Thinker Patcher/Merger**: Combines text, audio, and visual embeddings with DeepStack visual features across layers

#### 2. Talker (Speech Generation Module)

The Talker generates speech outputs from the Thinker's representations.

**Sub-models:**

- **Talker Embedding**: Converts input tokens to embeddings
- **Talker Hidden Projection**: Projects Thinker's hidden states to Talker's hidden space
- **Talker Text Projection**: Additional projection layer for text features
- **Talker Language Model**: MoE decoder (Qwen3MoeTalkerTextExperts) that processes projected features and generates codec predictions
- **Talker Code Predictor**: Predicts audio codec codes for speech synthesis (with separate embedding and decoder)

#### 3. Code2Wav Module

Converts predicted audio codes into waveform audio output (vocoder).

Let's convert each model part.

In [6]:
import ipywidgets as widgets

model_ids = ["Qwen/Qwen3-Omni-30B-A3B-Instruct", "Qwen/Qwen3-Omni-30B-A3B-Thinking"]

model_id = widgets.Dropdown(
    options=model_ids,
    default=model_ids[0],
    description="Model:",
)

model_id

Dropdown(description='Model:', options=('Qwen/Qwen3-Omni-30B-A3B-Instruct', 'Qwen/Qwen3-Omni-30B-A3B-Thinking'…

In [7]:
model_id = model_id.value
model_dir = Path(model_id.split("/")[-1])

### Compress Thinker Model Weights to 4 bits
[back to top ⬆️](#Table-of-contents:)

For reducing memory consumption, weights compression optimization can be applied using [NNCF](https://github.com/openvinotoolkit/nncf). 

<details>
    <summary><b>Click here for more details about weight compression</b></summary>
Weight compression aims to reduce the memory footprint of a model. It can also lead to significant performance improvement for large memory-bound models, such as Large Language Models (LLMs). LLMs and other models, which require extensive memory to store the weights during inference, can benefit from weight compression in the following ways:

* enabling the inference of exceptionally large models that cannot be accommodated in the memory of the device;

* improving the inference performance of the models by reducing the latency of the memory access when computing the operations with weights, for example, Linear layers.

[Neural Network Compression Framework (NNCF)](https://github.com/openvinotoolkit/nncf) provides 4-bit / 8-bit mixed weight quantization as a compression method primarily designed to optimize LLMs. The main difference between weights compression and full model quantization (post-training quantization) is that activations remain floating-point in the case of weights compression which leads to a better accuracy. Weight compression for LLMs provides a solid inference performance improvement which is on par with the performance of the full model quantization. In addition, weight compression is data-free and does not require a calibration dataset, making it easy to use.

`nncf.compress_weights` function can be used for performing weights compression. The function accepts an OpenVINO model and other compression parameters. Compared to INT8 compression, INT4 compression improves performance even more, but introduces a minor drop in prediction quality.

More details about weights compression, can be found in [OpenVINO documentation](https://docs.openvino.ai/2024/openvino-workflow/model-optimization-guide/weight-compression.html).

</details>

> **Note:** weights compression process may require additional time and memory for performing. You can disable it using widget below:

In [ ]:
import nncf
from qwen_3_omni_moe_helper import convert_qwen3_omni_moe_model

compression_configuration = {
    "mode": nncf.CompressWeightsMode.INT4_ASYM,
    "group_size": 128,
    "ratio": 0.8,
}

convert_qwen3_omni_moe_model(model_id, model_dir, compression_configuration)

## Prepare model inference pipeline
[back to top ⬆️](#Table-of-contents:)

As discussed, the Qwen3-Omni-MoE model comprises multiple specialized components including Vision Encoder, Audio Encoder, Thinker (understanding module), Talker (speech generation module), and Code2Wav vocoder. In `qwen_3_omni_moe_helper.py` we defined the Thinker inference class `OVQwen3OmniMoeThinkerForConditionalGeneration`, the Talker inference class `OVQwen3OmniMoeTalkerForConditionalGeneration`, and the Talker Code Predictor class `OVQwen3OmniMoeTalkerCodePredictorModelForConditionalGeneration` that represent the generation cycle. These classes are based on [HuggingFace Transformers `GenerationMixin`](https://huggingface.co/docs/transformers/main_classes/text_generation) and look similar to [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) `OVModelForCausalLM` used for LLM inference, with the key difference that they can accept multimodal input embeddings and support MoE architecture. The general multimodal model class `OVQwen3OmniMoeModel` orchestrates the entire pipeline, handling multimodal input processing (text, image, video, audio) and generating both text and speech outputs.

### Select device
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from notebook_utils import device_widget

thinker_device = device_widget(default="AUTO", exclude=["NPU"], description="Thinker device")

thinker_device

In [ ]:
talker_device = device_widget(default="AUTO", exclude=["NPU"], description="Talker device")

talker_device

In [ ]:
code2wav_device = device_widget(default="CPU", exclude=["NPU"], description="Code2Wav device")

code2wav_device

### Initialize model tasks
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from transformers import Qwen3OmniMoeProcessor
from qwen_3_omni_moe_helper import OVQwen3OmniMoeModel

ov_model = OVQwen3OmniMoeModel(model_dir, thinker_device=thinker_device.value, talker_device=talker_device.value, code2wav_device=code2wav_device.value)
processor = Qwen3OmniMoeProcessor.from_pretrained(model_dir)

## Run model inference
[back to top ⬆️](#Table-of-contents:)

Let's explore model capabilities using multimodal input and output.

In [ ]:
from qwen_omni_utils import process_mm_info
import soundfile as sf

conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-Omni/demo/cars.jpg"},
            {"type": "audio", "audio": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-Omni/demo/cough.wav"},
            {"type": "text", "text": "What can you see and hear? Answer in one short sentence."}
        ],
    },
]

# Set whether to use audio in video
USE_AUDIO_IN_VIDEO = True

# Preparation for inference
text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
audios, images, videos = process_mm_info(conversation, use_audio_in_video=USE_AUDIO_IN_VIDEO)
inputs = processor(text=text, 
                   audio=audios, 
                   images=images, 
                   videos=videos, 
                   return_tensors="pt", 
                   padding=True, 
                   use_audio_in_video=USE_AUDIO_IN_VIDEO)

# Inference: Generation of the output text and audio
text_ids, audio = ov_model.generate(**inputs, 
                                 speaker="Ethan", 
                                 thinker_return_dict_in_generate=True,
                                 use_audio_in_video=USE_AUDIO_IN_VIDEO)

text = processor.batch_decode(text_ids.sequences[:, inputs["input_ids"].shape[1] :],
                              skip_special_tokens=True,
                              clean_up_tokenization_spaces=False)
print(text)
if audio is not None:
    sf.write(
        "output_ov.wav",
        audio.reshape(-1).detach().cpu().numpy(),
        samplerate=24000,
    )


## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [15]:
from gradio_helper import make_demo

demo = make_demo(ov_model, processor)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/